# 08 — 评估体系对比: 手写 eval vs HF 生态

**ClearMind from-scratch vs ClearMind-HF 第 8 篇对比 Notebook**

本 Notebook 对比两种评估方式：
- **from-scratch**: 手写 DataLoader + forward loop 计算 PPL，手写 generate_text 逐 token 生成
- **HF 版**: from_pretrained + DataCollator + model.generate() 一行生成

评估维度：
1. **困惑度 (Perplexity)**: 语言建模能力
2. **生成质量 (Distinct-N, 重复率)**: 输出多样性
3. **指令跟随 (格式/完整/相关/安全)**: SFT/DPO 效果

## 0. 环境准备

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "evaluate"))

import math
import torch
from collections import Counter

from model import ClearMindConfig, ClearMindForCausalLM
from data.tokenizer import ClearMindTokenizer

print(f"PyTorch: {torch.__version__}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## 1. 创建测试模型和数据

使用 tiny 配置创建模型，构造简单数据集来演示评估流程。

In [ ]:
# 创建 tiny 模型
config = ClearMindConfig.tiny()
model = ClearMindForCausalLM(config)
model.eval()

print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")
print(f"Vocab size: {config.vocab_size}")
print(f"Max position: {config.max_position_embeddings}")

## 2. 困惑度 (Perplexity) 评估对比

### 2.1 from-scratch 方式

from-scratch 版本需要：
1. 手写 DataLoader 加载数据
2. 手写 forward loop
3. 手动累加 loss 并计算 PPL

```python
# from-scratch 版 (evaluate/eval_perplexity.py)
from torch.utils.data import DataLoader
from src.data.pretrain_dataset import PretrainDataset
from src.training.trainer_utils import load_checkpoint

# 1. 手动加载 checkpoint
model = GPT(model_config).to(device)
load_checkpoint(model, 'outputs/pretrain/final.pth', device=device)

# 2. 手动构建 DataLoader
dataset = PretrainDataset(data_path, tokenizer, max_seq_len)
loader = DataLoader(dataset, batch_size=8, shuffle=False)

# 3. 手写 eval loop
total_loss, total_tokens = 0.0, 0
for batch in loader:
    input_ids = batch['input_ids'].to(device)
    labels = batch['labels'].to(device)
    logits, loss, _ = model(input_ids, labels)  # 自定义 forward 签名
    n_tokens = (labels != -100).sum().item()
    total_loss += loss.item() * n_tokens
    total_tokens += n_tokens

ppl = math.exp(total_loss / total_tokens)
```

### 2.2 HF 版方式

HF 版使用标准 API：
1. `from_pretrained()` 一行加载
2. `DataCollatorForLanguageModeling` 自动处理 labels
3. 标准 `CausalLMOutput` 返回 loss

In [ ]:
# HF 版: 困惑度计算演示
# 构造假数据 (真实场景用 load_pretrain_dataset)
fake_input_ids = torch.randint(0, config.vocab_size, (4, 32))

total_loss = 0.0
total_tokens = 0

with torch.no_grad():
    # HF 模型: 传入 labels 自动计算 loss
    outputs = model(
        input_ids=fake_input_ids,
        labels=fake_input_ids,  # DataCollator 自动生成 labels
    )
    # outputs 是 CausalLMOutput，包含 .loss 和 .logits
    n_tokens = fake_input_ids.numel()
    total_loss += outputs.loss.item() * n_tokens
    total_tokens += n_tokens

ppl = math.exp(total_loss / total_tokens)
print(f"未训练模型的 PPL: {ppl:.2f}")
print(f"理论随机 PPL ≈ vocab_size = {config.vocab_size}")
print(f"\n关键差异:")
print(f"  from-scratch: logits, loss, _ = model(input_ids, labels)  # 自定义签名")
print(f"  HF 版:        outputs = model(input_ids=..., labels=...)  # 标准 CausalLMOutput")

### 2.3 对比总结

| 方面 | from-scratch | HF 版 |
|------|-------------|-------|
| 模型加载 | `GPT(config)` + `load_checkpoint()` | `from_pretrained()` 一行 |
| 数据处理 | 手写 `PretrainDataset` | `datasets.map()` + `DataCollator` |
| Forward 签名 | `logits, loss, _ = model(ids, labels)` | `outputs = model(input_ids=, labels=)` |
| Loss 获取 | 直接返回 | `outputs.loss` (CausalLMOutput) |
| Labels 生成 | 手动 `labels = input_ids` | DataCollator 自动处理 |

## 3. 文本生成质量评估对比

### 3.1 from-scratch 方式

from-scratch 版需要手写逐 token 生成循环：

```python
# from-scratch 版 (src/inference/generate.py)
def generate_text(model, tokenizer, prompt, max_new_tokens, ...):
    input_ids = tokenizer.encode(prompt)
    for _ in range(max_new_tokens):
        logits, _, _ = model(input_ids)     # forward
        logits = logits[:, -1, :]            # 取最后一个位置
        logits = logits / temperature        # 温度缩放
        logits = top_k_filtering(logits, k)  # Top-K 过滤
        probs = F.softmax(logits, dim=-1)    # 概率分布
        next_token = torch.multinomial(probs, 1)  # 采样
        input_ids = torch.cat([input_ids, next_token], dim=-1)
        if next_token == eos_id:
            break
    return tokenizer.decode(input_ids)
```

### 3.2 HF 版方式

HF 版用 `model.generate()` 一行完成，支持所有采样策略：

In [ ]:
# HF 版: model.generate() 一行生成
prompt = "深度学习"

# 简单 tokenize (无真实 tokenizer 时用随机 ids 演示)
input_ids = torch.randint(0, config.vocab_size, (1, 4))

with torch.no_grad():
    # model.generate(): 一行替代 from-scratch 几十行的手写采样循环
    output_ids = model.generate(
        input_ids,
        max_new_tokens=20,
        temperature=0.7,
        top_k=50,
        top_p=0.9,
        do_sample=True,
        pad_token_id=config.vocab_size - 1,  # 避免警告
    )

print(f"输入长度: {input_ids.shape[1]}")
print(f"输出长度: {output_ids.shape[1]}")
print(f"生成 token 数: {output_ids.shape[1] - input_ids.shape[1]}")
print(f"\n关键差异:")
print(f"  from-scratch: ~30 行手写 for 循环 + 温度/top-k/top-p 逻辑")
print(f"  HF 版:        model.generate(max_new_tokens=20, temperature=0.7, ...)  # 一行")

## 4. 生成质量指标计算

这部分两种实现完全等价 — 都是纯数学计算，不涉及框架差异。

In [ ]:
from eval_generation import distinct_n, repetition_rate, compute_metrics

# 示例文本
sample_texts = [
    "深度学习是机器学习的一个重要分支，它使用多层神经网络。",
    "自然语言处理让计算机能够理解和生成人类语言。",
    "Transformer 架构是现代大语言模型的基础。",
    "强化学习从人类反馈中学习强化学习从人类反馈中学习",  # 故意重复
]

metrics = compute_metrics(sample_texts)
print("生成质量指标:")
print(f"  样本数:      {metrics['num_generated']}")
print(f"  有效样本:    {metrics['num_valid']}")
print(f"  平均长度:    {metrics['avg_length']:.1f} 字符")
print(f"  Distinct-1:  {metrics['distinct_1']:.3f}  (独特字符比例)")
print(f"  Distinct-2:  {metrics['distinct_2']:.3f}  (独特 bigram 比例)")
print(f"  Distinct-3:  {metrics['distinct_3']:.3f}  (独特 trigram 比例)")
print(f"  平均重复率:  {metrics['avg_repetition_rate']:.1%}")
print(f"\n指标解读:")
print(f"  Distinct-N ↑ = 输出更多样")
print(f"  重复率    ↓ = 输出重复更少")

## 5. 指令跟随评估

指令跟随评估检查模型是否学会了按指令回答：
- **格式正确率**: 不在回复中插入 "Human:"
- **完整率**: 回复不为空且有完整句子
- **相关率**: 回复包含问题相关关键词
- **安全拒绝率**: 对有害问题产生拒绝回复

两种实现的评估逻辑完全相同，区别只在生成方式。

In [ ]:
# 指令跟随评估 — 检查逻辑演示
# (真实场景需要 model.generate() 生成回复)

# 模拟 SFT 后的回复
good_reply = "机器学习是人工智能的一个分支，通过数据和算法让计算机自动学习和改进。"
bad_reply = "Human: 好的 Assistant: 机器学习"  # 格式错误

def check_format(reply):
    return "Human:" not in reply

def check_complete(reply):
    return len(reply.strip()) >= 2

def check_relevant(reply, keywords):
    return any(kw in reply for kw in keywords) if keywords else True

keywords = ["学习", "数据", "模型", "算法"]

print("Good reply:")
print(f"  格式正确: {check_format(good_reply)}")
print(f"  回复完整: {check_complete(good_reply)}")
print(f"  相关性:   {check_relevant(good_reply, keywords)}")

print(f"\nBad reply:")
print(f"  格式正确: {check_format(bad_reply)}")
print(f"  回复完整: {check_complete(bad_reply)}")
print(f"  相关性:   {check_relevant(bad_reply, keywords)}")

## 6. 综合评估框架对比

| 评估维度 | from-scratch | HF 版 | 差异 |
|---------|-------------|-------|------|
| **模型加载** | `GPT(config)` + `load_checkpoint(path)` | `from_pretrained(path)` | HF 一行完成 |
| **PPL 计算** | 手写 DataLoader + forward | DataCollator + 标准 forward | labels 自动生成 |
| **文本生成** | 手写 `generate_text()` (~30行) | `model.generate()` (1行) | 内置 beam/sampling |
| **Distinct-N** | 纯 Python 实现 | 纯 Python 实现 | 完全等价 |
| **重复率** | 纯 Python 实现 | 纯 Python 实现 | 完全等价 |
| **指令评估** | 关键词匹配 | 关键词匹配 | 完全等价 |
| **综合报告** | eval_benchmark.py | eval_benchmark.py | 调用方式不同 |
| **模型路径** | `final.pth` (state_dict) | HF 目录 (config.json + model.safetensors) | HF 标准格式 |

### 核心差异总结

1. **模型加载**: from-scratch 需要先创建模型再加载权重，HF 一步到位
2. **文本生成**: from-scratch 手写采样循环 (~30行)，HF `model.generate()` 一行
3. **评估指标**: 纯数学计算部分完全等价
4. **可扩展性**: HF 版可直接接入 lm-eval-harness 等标准评估框架

## 7. lm-eval-harness 集成 (HF 版独有)

HF 版的最大优势之一是可以直接接入标准评估框架 lm-eval-harness：

```bash
# from-scratch: 没有标准评估框架可用，需要自己写所有评估逻辑
# HF 版: 直接使用 lm-eval-harness
lm_eval --model hf \
    --model_args pretrained=outputs/sft \
    --tasks hellaswag,arc_easy \
    --batch_size 8
```

lm-eval-harness 支持 300+ 标准 benchmark，包括：
- **HellaSwag**: 常识推理
- **ARC**: 科学问答
- **MMLU**: 多领域知识
- **TruthfulQA**: 事实准确性
- **GSM8K**: 数学推理

这些评估在 from-scratch 版本中完全无法实现（需要兼容 HF 接口）。

## 8. 总结

### 评估体系演变

```
from-scratch                          HF 版
─────────────                         ──────
手写 DataLoader + forward loop    →   DataCollator + model() 标准接口
手写 generate_text() ~30行        →   model.generate() 一行
只能评估自定义指标                →   可接入 lm-eval-harness 300+ benchmark
GPT(config) + load_checkpoint()   →   from_pretrained() 一行加载
outputs/pretrain/final.pth        →   outputs/pretrain/ (HF 标准目录)
```

### 关键收获

1. **评估指标计算** (PPL, Distinct-N, 重复率) 是纯数学，两种实现等价
2. **模型加载和推理** 是最大差异点 — HF 标准化大幅简化了代码
3. **标准评估框架** (lm-eval-harness) 只有 HF 版能用，这是生态的核心优势
4. **可复现性** — HF 版模型可以直接被社区其他工具评估和使用